In [ ]:
import os, shutil, datetime, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import pandas as pd
from pathlib import Path
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_Aug4000_Results"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)

In [ ]:
IMG_SIZE_RAW = (48, 48)    # dataset images (grayscale)
IMG_SIZE_INP = (224, 224)  # model input
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE


def make_loader(dir_path, shuffle=True, batch=BATCH):
    ds = image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="categorical",   # one-hot for Keras metrics/ROC
        color_mode="grayscale",
        image_size=IMG_SIZE_RAW,
        batch_size=batch,
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_INP)
        x = tf.cast(x, tf.float32) / 255.0  # keep [0,1]; branch-wise preprocess later
        return x, y

    return ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), class_names

train_ds, class_names = make_loader(LOCAL_TRAIN, shuffle=True)
val_ds, _             = make_loader(LOCAL_VAL,   shuffle=False)
test_ds, _            = make_loader(LOCAL_TEST,  shuffle=False)
num_classes = len(class_names)
print("Classes:", class_names, "| Num classes:", num_classes)


Found 28000 files belonging to 7 classes.
Found 8616 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'] | Num classes: 7


In [ ]:
# ---------------- Hybrid model: ResNet101 + MobileNetV3 feature fusion ----------------
from tensorflow.keras.applications.resnet import preprocess_input as resnet_pp
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mnetv3_pp
from tensorflow.keras.applications import ResNet101, MobileNetV3Large

# Backbones (frozen)
resnet_base = ResNet101(include_top=False, weights="imagenet", input_shape=(224,224,3))
mnet_base   = MobileNetV3Large(include_top=False, weights="imagenet", input_shape=(224,224,3))
resnet_base.trainable = False
mnet_base.trainable   = False

inp = layers.Input(shape=(224,224,3), name="inp_rgb01")  # [0,1] RGB

# Branch-1: ResNet101 preprocess & features
x_r = layers.Lambda(lambda z: resnet_pp(z*255.0), name="resnet_preproc")(inp)
f_r = resnet_base(x_r, training=False)
f_r = layers.GlobalAveragePooling2D(name="gap_resnet101")(f_r)

# Branch-2: MobileNetV3 preprocess & features
x_m = layers.Lambda(lambda z: mnetv3_pp(z*255.0), name="mnetv3_preproc")(inp)
f_m = mnet_base(x_m, training=False)
f_m = layers.GlobalAveragePooling2D(name="gap_mobilenetv3")(f_m)

# Fusion + classifier head
f = layers.Concatenate(name="concat_feats")([f_r, f_m])
f = layers.BatchNormalization()(f)
f = layers.Dense(512, activation="relu")(f)
f = layers.BatchNormalization()(f)
f = layers.Dropout(0.5)(f)
out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

model = models.Model(inp, out, name="Hybrid_ResNet101_MobileNetV3")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "Hybrid_ResNet101_MobileNetV3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_rgb01           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_preproc      │ (None, 224, 224,  │          0 │ inp_rgb01[0][0]   │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mnetv3_preproc      │ (None, 224, 224,  │          0 │ inp_rgb01[0][0]   │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet101           │ (None, 7, 7,      │ 42,658,176 │ resnet_preproc[0… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MobileNetV3Large    │ (None, 7, 7, 960) │  2,996,352 │ mnetv3_preproc[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap_resnet101       │ (None, 2048)      │          0 │ resnet101[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap_mobilenetv3     │ (None, 960)       │          0 │ MobileNetV3Large… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_feats        │ (None, 3008)      │          0 │ gap_resnet101[0]… │
│ (Concatenate)       │                   │            │ gap_mobilenetv3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 3008)      │     12,032 │ concat_feats[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,540,608 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pred (Dense)        │ (None, 7)         │      3,591 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 47,212,807 (180.10 MB)

 Trainable params: 1,551,239 (5.92 MB)

 Non-trainable params: 45,661,568 (174.19 MB)

In [ ]:
# ---------------- Callbacks ----------------
ckpt_path = f"{OUT_DIR}/best_model.keras"  # use native Keras format
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

In [ ]:
# ---------------- Train ----------------
EPOCHS = 20
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/20
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.3626 - loss: 2.1709
Epoch 1: val_accuracy improved from -inf to 0.52739, saving model to /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_Aug4000_Results/best_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 278s 277ms/step - accuracy: 0.3627 - loss: 2.1706 - val_accuracy: 0.5274 - val_loss: 1.3661 - learning_rate: 1.0000e-04
Epoch 2/20
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step - accuracy: 0.5228 - loss: 1.4375
Epoch 2: val_accuracy improved from 0.52739 to 0.55327, saving model to /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_Aug4000_Results/best_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 214s 245ms/step - accuracy: 0.5228 - loss: 1.4375 - val_accuracy: 0.5533 - val_loss: 1.2746 - learning_rate: 1.0000e-04
Epoch 3/20
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step - accuracy: 0.5702 - loss: 1.2429
Epoch 3: val_accuracy improved from 0.55327 to 0.56639, savin

In [ ]:
# Save model & history
model.save(os.path.join(OUT_DIR, "hybrid_resnet101_mnetv3_final.keras"))
pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)


In [ ]:
# ---------------- Curves ----------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy Curve"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
plt.close()
print("Training curves saved.")


Training curves saved.


In [ ]:
# ---------------- Evaluate on Test ----------------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")


TEST -> loss: 1.2402 | acc: 0.5706


In [ ]:
# ---------------- Predictions & Reports ----------------
# true labels (one-hot)
y_true = []
for _, y in test_ds.unbatch():
    y_true.append(y.numpy())
y_true = np.array(y_true)
y_true_cls = np.argmax(y_true, axis=1)

# probs & preds
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Classification Report
rep = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
print(rep)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(rep)


              precision    recall  f1-score   support

       angry     0.4664    0.4562    0.4612       958
   disgusted     0.5146    0.4775    0.4953       111
     fearful     0.4547    0.3770    0.4122      1024
       happy     0.7631    0.7610    0.7621      1774
     neutral     0.5187    0.5629    0.5399      1233
         sad     0.4377    0.4651    0.4510      1247
   surprised     0.6954    0.7172    0.7062       831

    accuracy                         0.5706      7178
   macro avg     0.5501    0.5453    0.5468      7178
weighted avg     0.5693    0.5706    0.5692      7178



In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix"); plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=45, ha="right")
plt.yticks(ticks, class_names)
th = cm.max()/2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > th else "black", fontsize=8)
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
print("Confusion matrix saved.")

# ---------------- ROC Curves (OvR + micro) ----------------
y_true_bin = y_true  # already one-hot
num_classes = y_true_bin.shape[1]
fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8,6))
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average ROC (AUC = {roc_auc['micro']:.3f})", linewidth=2)
colors = ['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown']
for i, cname in enumerate(class_names):
    plt.plot(fpr[i], tpr[i], lw=2, color=colors[i % len(colors)], label=f"{cname} (AUC = {roc_auc[i]:.3f})")
plt.plot([0,1], [0,1], "k--", lw=1)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC (One-vs-Rest)")
plt.legend(loc="lower right", fontsize=8)
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"), dpi=150)
plt.close()
print("ROC curves saved:", os.path.join(OUT_DIR, "roc_curves.png"))

print("✅ All outputs saved to:", OUT_DIR)

Confusion matrix saved.
ROC curves saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_Aug4000_Results/roc_curves.png
✅ All outputs saved to: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_Aug4000_Results
